# 🧠 Pipeline RAG & Architecture SMA — Study-Buddy

**Projet SMA Agentic AI — EMSI**  
**Étudiantes :** EL OUALFI Meriem & BOUTARFA Wafae  

---

Ce notebook documente l'architecture complète du **Système Multi-Agents (SMA)** Study-Buddy,  
incluant la pipeline RAG, le routage intelligent et la justification des choix techniques.

## Architecture Globale SMA

```
                        ┌─────────────────────────────────┐
                        │         FRONTEND                │
                        │   Interface StudyBuddy          │
                        │   (question + fileName)         │
                        └────────────┬────────────────────┘
                                     │ POST webhook
                                     ▼
                        ┌────────────────────────┐
                        │   Webhook n8n          │
                        │   /webhook/study-buddy │
                        └────────────┬───────────┘
                                     │
                                     ▼
                        ┌────────────────────────┐
                        │   Edit Fields          │
                        │   question, fileName,  │
                        │   sessionId, email     │
                        └────────────┬───────────┘
                                     │
                                     ▼
                        ┌────────────────────────┐
                        │   Agent Manager        │  ← ROUTEUR
                        │   (LLM Llama 3.1 8B)  │
                        │   Classifie: SOCRATIQUE│
                        │   QUIZ / RESUME /      │
                        │   EVALUATION / ACCUEIL │
                        └────────────┬───────────┘
                                     │
                         ┌───────────┴───────────┐
                         │   Switch - Router     │
                         └──┬──────┬──────┬──────┘
                            │      │      │      
                     ┌──────┘  ┌───┘  ┌──┘  
                     ▼         ▼      ▼     
              SOCRATIQUE    QUIZ   RESUME  EVALUATION
              Agent +       Agent  Agent   Agent
              Pinecone      +Pine  +Pine   +Pine
                     └──────┴───────┴────────┘
                                  │
                                  ▼
                        ┌─────────────────┐
                        │ Merge Responses │
                        └────────┬────────┘
                                 │
                                 ▼
                        ┌─────────────────┐
                        │ Respond Webhook │
                        └─────────────────┘
```


## 1. Configuration

In [1]:
import requests
import json
import time

# Webhook principal Study-Buddy
N8N_WEBHOOK_URL = "https://sttudybuddy.app.n8n.cloud/webhook/study-buddy"

# Configuration du SMA
CONFIG = {
    "model": "meta-llama/llama-3.1-8b-instruct",
    "pinecone_index": "studybuddy-index",
    "embedding_model": "Google Gemini Embeddings",
    "agents": ["SOCRATIQUE", "QUIZ", "RESUME", "EVALUATION"],
    "memory": "Window Buffer Memory (n8n)"
}

print("✅ Configuration SMA Study-Buddy")
for k, v in CONFIG.items():
    print(f"   {k:20} : {v}")

✅ Configuration SMA Study-Buddy
   model                : meta-llama/llama-3.1-8b-instruct
   pinecone_index       : studybuddy-index
   embedding_model      : Google Gemini Embeddings
   agents               : ['SOCRATIQUE', 'QUIZ', 'RESUME', 'EVALUATION']
   memory               : Window Buffer Memory (n8n)


## 2. Justification du SMA — Pourquoi 3+ agents et non un seul prompt ?

### Problème avec un seul prompt massif

| Critère | 1 Prompt Massif | SMA (notre approche) |
|---------|----------------|----------------------|
| Context window | Explose rapidement | Chaque agent gère son contexte |
| Qualité réponse | Dégradée (tâches mixées) | Optimisée par spécialisation |
| Maintenance | Difficile | Chaque agent modifiable indépendamment |
| Coût tokens | Très élevé | Réduit (prompt court par agent) |
| Fiabilité | Hallucinations fréquentes | RAG ancre les réponses dans le cours |

### Nos 4 agents spécialisés

1. **Agent Manager** — Routeur : analyse la question et choisit l'agent approprié
2. **Agent Socratique** — Tuteur : guide l'étudiant par questions sans donner la réponse
3. **Agent Quiz** — Évaluateur : génère des QCM basés sur le cours
4. **Agent Resume** — Synthétiseur : crée des fiches de révision structurées
5. **Agent Evaluation** — Correcteur : évalue et note les réponses de l'étudiant

## 3. Test du pipeline RAG complet

In [5]:
def appeler_study_buddy(question, file_name="", session_id=None, email="meriem.eloualfi1@gmail.com", name="EL OUALFI Meriem"):
    """
    Appelle le webhook Study-Buddy et retourne la réponse de l'agent.

    Args:
        question: La question ou commande de l'étudiant
        file_name: Nom du fichier cours uploadé (namespace Pinecone)
        session_id: ID de session pour la mémoire conversationnelle
        email: Email de l'étudiant
        name: Nom de l'étudiant

    Returns:
        dict: Réponse complète avec mode, texte et métadonnées
    """
    if session_id is None:
        session_id = f"session-{int(time.time())}"

    payload = {
        "question": question,
        "fileName": file_name,
        "sessionId": session_id,
        "email": email,
        "name": name
    }

    print(f"📤 Question : {question}")
    print(f"   Fichier  : {file_name if file_name else 'Aucun (connaissance générale)'}")

    debut = time.time()
    response = requests.post(N8N_WEBHOOK_URL, json=payload)
    duree = round(time.time() - debut, 2)

    if response.status_code == 200:
        result = response.json()
        mode = result.get('mode', 'INCONNU')
        texte = result.get('text', '')

        print(f"\n✅ Réponse reçue en {duree}s")
        print(f"   Mode    : {mode}")
        print(f"   Réponse : {texte[:200]}..." if len(texte) > 200 else f"   Réponse : {texte}")
        return result
    else:
        print(f"❌ Erreur {response.status_code}: {response.text}")
        return None

print("✅ Fonction appeler_study_buddy() prête")

✅ Fonction appeler_study_buddy() prête


## 4. Test Agent Socratique (avec cours RAG)

In [11]:
print("=" * 60)
print("TEST 1 — Agent SOCRATIQUE avec cours")
print("=" * 60)

result_socratique = appeler_study_buddy(
    question="Explique ce concept",
    file_name="chapitre3_IOT.pdf",
    session_id="test-socratique-001"
)

print("\n📋 Réponse complète :")
print(json.dumps(result_socratique, ensure_ascii=False, indent=2) if result_socratique else "Aucune réponse")

TEST 1 — Agent SOCRATIQUE avec cours
📤 Question : Explique ce concept
   Fichier  : chapitre3_IOT.pdf

✅ Réponse reçue en 5.12s
   Mode    : SOCRATIQUE
   Réponse : [MODE: SOCRATIQUE]
Le concept que tu as mentionné n'est pas clair. Pourrait-il te falloir plus de détails?

Qu'est-ce que tu asimer apprendre sur ce concept?

📋 Réponse complète :
{
  "text": "[MODE: SOCRATIQUE]\nLe concept que tu as mentionné n'est pas clair. Pourrait-il te falloir plus de détails?\n\nQu'est-ce que tu asimer apprendre sur ce concept?",
  "mode": "SOCRATIQUE",
  "success": true,
  "timestamp": "2026-05-20T08:24:20.022Z"
}


## 5. Test Agent QUIZ

In [7]:
print("=" * 60)
print("TEST 2 — Agent QUIZ")
print("=" * 60)

result_quiz = appeler_study_buddy(
    question="Je veux un quiz sur ce cours",
    file_name="chapitre3_IOT.pdf",
    session_id="test-socratique-001"
)

# Extraire les questions du quiz
if result_quiz and result_quiz.get('mode') == 'QUIZ':
    print("\n✅ Quiz généré avec succès !")
    questions = result_quiz.get('questions', [])
    print(f"   Nombre de questions : {len(questions)}")
    for i, q in enumerate(questions[:2], 1):  # Afficher les 2 premières
        print(f"\n   Q{i}: {q.get('q', '')}")
        for j, opt in enumerate(q.get('options', [])):
            marker = "✓" if j == q.get('answer', -1) else " "
            print(f"      [{marker}] {opt}")

TEST 2 — Agent QUIZ
📤 Question : Je veux un quiz sur ce cours
   Fichier  : chapitre3_IOT.pdf

✅ Réponse reçue en 18.18s
   Mode    : QUIZ
   Réponse : 

✅ Quiz généré avec succès !
   Nombre de questions : 5

   Q1: Qu'est-ce que l'IOT?
      [✓] Un système de communication sans fil
      [ ] Un système de stockage de données
      [ ] Un système de sécurité informatique
      [ ] Un système de gestion de projet

   Q2: Quel est le principal avantage de l'IOT?
      [✓] Meilleure efficacité
      [ ] Meilleure sécurité
      [ ] Meilleure productivité
      [ ] Meilleure qualité de vie


## 6. Test Agent RESUME

In [8]:
print("=" * 60)
print("TEST 3 — Agent RESUME")
print("=" * 60)

result_resume = appeler_study_buddy(
    question="Fais moi un résumé du cours",
    file_name="chapitre3_IOT.pdf",
    session_id="test-socratique-001"
)

if result_resume:
    print(f"\n✅ Mode détecté : {result_resume.get('mode')}")
    print(f"   Résumé généré : {len(result_resume.get('text', ''))} caractères")

TEST 3 — Agent RESUME
📤 Question : Fais moi un résumé du cours
   Fichier  : chapitre3_IOT.pdf

✅ Réponse reçue en 9.04s
   Mode    : RESUME
   Réponse : [MODE: RESUME]
TITRE : Résumé du cours
POINTS CLES :
- Introduction à l'IoT
- Avantages et inconvénients de l'IoT
- Exemples concrets d'applications de l'IoT
A RETENIR : L'IoT est un domaine qui se dé...

✅ Mode détecté : RESUME
   Résumé généré : 378 caractères


## 7. Test Agent EVALUATION

In [14]:
print("=" * 60)
print("TEST 4 — Agent EVALUATION")
print("=" * 60)

result_eval = appeler_study_buddy(
    question="Voici ma réponse : L'IoT connecte des objets physiques à Internet via des protocoles comme MQTT et CoAP. Les capteurs collectent des données envoyées vers le cloud pour traitement.",
    file_name="chapitre3_IOT.pdf",
    session_id="test-eval-001"   # ← session différente des autres tests
)

if result_eval:
    print(f"\n✅ Mode détecté : {result_eval.get('mode')}")
    print(f"   Evaluation : {result_eval.get('text', '')[:300]}")

TEST 4 — Agent EVALUATION
📤 Question : Voici ma réponse : L'IoT connecte des objets physiques à Internet via des protocoles comme MQTT et CoAP. Les capteurs collectent des données envoyées vers le cloud pour traitement.
   Fichier  : chapitre3_IOT.pdf

✅ Réponse reçue en 4.97s
   Mode    : SOCRATIQUE
   Réponse : [MODE: SOCRATIQUE]
Il semble que les capteurs collectent des données puis les aident à améliorer notre vie quotidienne. Mais qu'est-ce que vous entendez par "traitement" des données sur le cloud?

✅ Mode détecté : SOCRATIQUE
   Evaluation : [MODE: SOCRATIQUE]
Il semble que les capteurs collectent des données puis les aident à améliorer notre vie quotidienne. Mais qu'est-ce que vous entendez par "traitement" des données sur le cloud?


## 8. Gestion du Fallback et des Hallucinations

### Stratégies implémentées :

**1. Fallback Pinecone vide**  
Si Pinecone ne trouve aucun chunk pertinent, l'agent utilise sa connaissance générale (activé via `Continue on Fail` sur le node Pinecone).

**2. Fallback agent**  
Si tous les agents échouent, un node `Fallback - Message Erreur` retourne un message clair à l'utilisateur.

**3. Anti-hallucination RAG**  
Les agents sont instruits de répondre UNIQUEMENT depuis le contenu récupéré par Pinecone, avec des prompts stricts du type :
```
"Génère des questions basées UNIQUEMENT sur le contenu récupéré.
Ne JAMAIS inventer du contenu non présent dans le cours."
```

**4. Validation JSON Quiz**  
Le node `Valider JSON Quiz` vérifie que le format retourné est valide avant d'envoyer la réponse.  
En cas d'échec → `Respond - Quiz Self-Correction` relance l'agent avec instructions de correction.

**5. Enregistrement Gmail**  
Chaque évaluation est envoyée par email via Gmail (node `Gmail - Envoyer Fiche Memo`) pour traçabilité.

In [15]:
# Résumé des stratégies anti-hallucination
strategies = [
    ("RAG Grounding", "Les réponses sont ancrées dans le contenu Pinecone"),
    ("Prompts stricts", "Instructions explicites: UNIQUEMENT le contenu du cours"),
    ("Continue on Fail", "Pinecone vide → connaissance générale plutôt qu'erreur"),
    ("Fallback node", "Si tous agents échouent → message d'erreur clair"),
    ("Validation JSON", "Format Quiz vérifié avant envoi"),
    ("Self-correction", "Agent Quiz se relance si format invalide"),
    ("Gmail logging", "Traçabilité des évaluations par email")
]

print("🛡️  Stratégies anti-hallucination & fallback")
print("=" * 50)
for nom, description in strategies:
    print(f"  ✅ {nom:20} : {description}")

🛡️  Stratégies anti-hallucination & fallback
  ✅ RAG Grounding        : Les réponses sont ancrées dans le contenu Pinecone
  ✅ Prompts stricts      : Instructions explicites: UNIQUEMENT le contenu du cours
  ✅ Continue on Fail     : Pinecone vide → connaissance générale plutôt qu'erreur
  ✅ Fallback node        : Si tous agents échouent → message d'erreur clair
  ✅ Validation JSON      : Format Quiz vérifié avant envoi
  ✅ Self-correction      : Agent Quiz se relance si format invalide
  ✅ Gmail logging        : Traçabilité des évaluations par email


## 9. Analyse de Coût — Gestion des tokens

| Agent | Tokens prompt moyen | Tokens réponse | Coût estimé/appel |
|-------|--------------------|-----------------|-----------------|
| Manager | ~200 | ~10 | ~$0.0001 |
| Socratique | ~800 | ~100 | ~$0.0004 |
| Quiz | ~1200 | ~600 | ~$0.0009 |
| Resume | ~1500 | ~800 | ~$0.0012 |
| Evaluation | ~1000 | ~300 | ~$0.0007 |

**Total par session typique :** ~$0.002  
**Modèle utilisé :** meta-llama/llama-3.1-8b-instruct (via OpenAI-compatible API)  
**Optimisation :** Seul 1 agent est appelé par requête (pas tous en parallèle)

In [16]:
# Calcul du coût estimé
agents_cout = {
    "Manager": {"prompt": 200, "response": 10},
    "Socratique": {"prompt": 800, "response": 100},
    "Quiz": {"prompt": 1200, "response": 600},
    "Resume": {"prompt": 1500, "response": 800},
    "Evaluation": {"prompt": 1000, "response": 300},
}

# Llama 3.1 8B pricing (approximatif)
PRIX_1M_TOKENS = 0.10  # $0.10 par million de tokens

print("💰 Analyse de coût par agent")
print("-" * 55)

total_tokens = 0
for agent, tokens in agents_cout.items():
    total = tokens['prompt'] + tokens['response']
    cout = (total / 1_000_000) * PRIX_1M_TOKENS
    print(f"  {agent:15} : {total:5} tokens → ${cout:.6f}")
    total_tokens += total

cout_total = (total_tokens / 1_000_000) * PRIX_1M_TOKENS
print("-" * 55)
print(f"  {'TOTAL (worst)':15} : {total_tokens:5} tokens → ${cout_total:.6f}")
print(f"\n  ✅ Coût réel par appel : ~${cout_total/5:.6f} (1 agent seulement)")

💰 Analyse de coût par agent
-------------------------------------------------------
  Manager         :   210 tokens → $0.000021
  Socratique      :   900 tokens → $0.000090
  Quiz            :  1800 tokens → $0.000180
  Resume          :  2300 tokens → $0.000230
  Evaluation      :  1300 tokens → $0.000130
-------------------------------------------------------
  TOTAL (worst)   :  6510 tokens → $0.000651

  ✅ Coût réel par appel : ~$0.000130 (1 agent seulement)
